<a href="https://colab.research.google.com/github/pathianil40/ML_PROJECTS/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score



In [ ]:
df=pd.read_csv('/content/archive (1).zip')

In [ ]:
print(df.head())

   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0      100002       1         Cash loans           M            N   
1      100003       0         Cash loans           F            N   
2      100004       0    Revolving loans           M            Y   
3      100006       0         Cash loans           F            N   
4      100007       0         Cash loans           M            N   

  FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0               Y             0          202500.0    406597.5      24700.5   
1               N             0          270000.0   1293502.5      35698.5   
2               Y             0           67500.0    135000.0       6750.0   
3               Y             0          135000.0    312682.5      29686.5   
4               Y             0          121500.0    513000.0      21865.5   

   ...  FLAG_DOCUMENT_18 FLAG_DOCUMENT_19 FLAG_DOCUMENT_20 FLAG_DOCUMENT_21  \
0  ...                 0             

In [ ]:
print(df.tail())

        SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
307506      456251       0         Cash loans           M            N   
307507      456252       0         Cash loans           F            N   
307508      456253       0         Cash loans           F            N   
307509      456254       1         Cash loans           F            N   
307510      456255       0         Cash loans           F            N   

       FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  \
307506               N             0          157500.0    254700.0   
307507               Y             0           72000.0    269550.0   
307508               Y             0          153000.0    677664.0   
307509               Y             0          171000.0    370107.0   
307510               N             0          157500.0    675000.0   

        AMT_ANNUITY  ...  FLAG_DOCUMENT_18 FLAG_DOCUMENT_19 FLAG_DOCUMENT_20  \
307506      27558.0  ...                 0            

In [ ]:
X=df.drop(columns='TARGET',axis=1)
y=df['TARGET']

In [ ]:
print("no of records:",X.shape[1])

no of records: 121


In [ ]:
print("no of samples:",X.shape[0])

no of samples: 307511


In [ ]:
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_columns = X.select_dtypes(
    include=["object"]
).columns


In [ ]:
print("numerical_columns:",numerical_columns)
print("categorical_columns:",categorical_columns)

numerical_columns: Index(['SK_ID_CURR', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
       'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE',
       'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
       ...
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR',
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR'],
      dtype='object', length=105)
categorical_columns: Index(['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
       'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
       'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE',
       'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE'],
      dtype='object')


In [ ]:
numerical_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='mean')),
    ('scaler',StandardScaler())
])
categorical_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=True))
])

In [ ]:
preprocessor=ColumnTransformer(transformers=[
    ('num',numerical_pipeline,numerical_columns),
    ('cat',categorical_pipeline,categorical_columns)
])

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print("training records:",X_train.shape[0])
print("testing records:",X_test.shape[0])

training records: 246008
testing records: 61503


In [ ]:
random_forest=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('regressor',RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1))
])
gb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            GradientBoostingRegressor(n_estimators=100, random_state=42),
        ),
    ]
)

# 7. Train & Time Random Forest
print("Starting Random Forest training...")
start_time = time.time()
random_forest.fit(X_train, y_train)
rf_training_time = time.time() - start_time
print(f"Random Forest Training Time: {rf_training_time:.2f} seconds")

# 8. Train & Time Gradient Boosting
print("\nStarting Gradient Boosting training...")
start_time = time.time()
gb_model.fit(X_train, y_train)
gb_training_time = time.time() - start_time
print(f"Gradient Boosting Training Time: {gb_training_time:.2f} seconds")

# 9. Make Predictions
rf_preds = random_forest.predict(X_test)
gb_preds = gb_model.predict(X_test)

# 10. Evaluation Metrics
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_mse = mean_squared_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)

gb_mae = mean_absolute_error(y_test, gb_preds)
gb_mse = mean_squared_error(y_test, gb_preds)
gb_r2 = r2_score(y_test, gb_preds)

# 11. Print Comparison Report
print("\n" + "=" * 45)
print("           MODEL COMPARISON REPORT           ")
print("=" * 45)
print(f"{'Metric':<20} | {'Random Forest':<10} | {'Gradient Boosting':<10}")
print("-" * 45)
print(
    f"{'Training Time (s)':<20} | {rf_training_time:<10.2f} | {gb_training_time:<10.2f}"
)
print(f"{'MAE':<20} | {rf_mae:<10.4f} | {gb_mae:<10.4f}")
print(f"{'MSE':<20} | {rf_mse:<10.4f} | {gb_mse:<10.4f}")
print(f"{'R2 Score':<20} | {rf_r2:<10.4f} | {gb_r2:<10.4f}")
print("=" * 45)

# Summary Insights
if gb_r2 > rf_r2:
    print("\nOutcome: Gradient Boosting achieved a higher R² score.")
elif rf_r2 > gb_r2:
    print("\nOutcome: Random Forest achieved a higher R² score.")
else:
    print("\nOutcome: Both models yielded equal R² performance.")


Starting Random Forest training...
Random Forest Training Time: 2087.90 seconds

Starting Gradient Boosting training...
Gradient Boosting Training Time: 548.72 seconds

           MODEL COMPARISON REPORT           
Metric               | Random Forest | Gradient Boosting
---------------------------------------------
Training Time (s)    | 2087.90    | 548.72    
MAE                  | 0.1482     | 0.1385    
MSE                  | 0.0714     | 0.0683    
R2 Score             | 0.0355     | 0.0775    

Outcome: Gradient Boosting achieved a higher R² score.
